# ExtremeTrack One-Go Trainer (Kaggle)

This notebook is configured for Kaggle training.
Attach the ExtremeTrack dataset in the right sidebar and it will auto-detect the folder under `/kaggle/input/...`.

It trains a restoration-assisted MixFormer-style tracker and validates with official QP.

In [3]:
!pip -q install brisque ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.0 MB/s eta 0:00:0000:01


In [4]:
import os
import gc
import json
import math
import time
import random
from dataclasses import dataclass
from pathlib import Path
from typing import List

import cv2
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import torchvision.transforms.functional as TF

from brisque import BRISQUE

# ------------------------------
# Runtime + path config
# ------------------------------
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
LOCAL_DATASET_ROOT = Path('./ExtremeTrack')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type != 'cuda':
    raise RuntimeError('Enable GPU runtime first.')

IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ


def find_dataset_root() -> Path:
    candidates = [
        KAGGLE_INPUT_ROOT,
        Path('/kaggle/working'),
        LOCAL_DATASET_ROOT,
        Path('.'),
    ]

    required = ('ExtremeTrack_train.json', 'ExtremeTrack_val.json')

    for c in candidates:
        if all((c / name).exists() for name in required):
            return c

    for base in candidates:
        if not base.exists():
            continue
        try:
            for p in base.rglob('ExtremeTrack_train.json'):
                root = p.parent
                if (root / 'ExtremeTrack_val.json').exists():
                    return root
        except Exception:
            continue

    raise FileNotFoundError(
        'ExtremeTrack_train.json and ExtremeTrack_val.json were not found. '
        'In Kaggle, add the ExtremeTrack dataset from the Input sidebar first.'
    )


DATASET_ROOT = find_dataset_root()
print('DATASET_ROOT =', DATASET_ROOT)

OUTPUT_ROOT = Path('/kaggle/working/extremetrack_outputs') if IN_KAGGLE else Path('./extremetrack_outputs')
(OUTPUT_ROOT / 'checkpoints').mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / 'predictions').mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / 'experiments').mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / 'cache').mkdir(parents=True, exist_ok=True)
print('OUTPUT_ROOT =', OUTPUT_ROOT)

Device: cuda


/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


FileNotFoundError: ExtremeTrack_train.json and ExtremeTrack_val.json not found.

In [ ]:
def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def save_json(path: Path, payload: dict):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)


@dataclass
class SequenceRecord:
    name: str
    video_dir: str
    image_paths: List[str]
    gt_rect: List[List[float]]
    condition: str


def load_sequences(annotation_name: str, condition_filter='all'):
    obj = load_json(DATASET_ROOT / annotation_name)
    seqs = []
    for name, seq in obj.items():
        condition = 'haze' if name.lower().endswith('haze') else 'rain'
        if condition_filter != 'all' and condition != condition_filter:
            continue
        seqs.append(SequenceRecord(
            name=name,
            video_dir=seq['video_dir'],
            image_paths=seq['img_names'],
            gt_rect=seq['gt_rect'],
            condition=condition,
        ))
    return seqs


def clip_box_xywh(box, width, height, min_size=2.0):
    x, y, w, h = box
    w = max(min_size, min(w, width - 1))
    h = max(min_size, min(h, height - 1))
    x = min(max(0.0, x), max(0.0, width - w))
    y = min(max(0.0, y), max(0.0, height - h))
    return [float(x), float(y), float(w), float(h)]


def compute_square_crop(box, scale):
    x, y, w, h = box
    cx, cy = x + w / 2.0, y + h / 2.0
    side = max(w, h) * scale
    side = max(side, 8.0)
    return cx, cy, side


def crop_square(image: Image.Image, cx: float, cy: float, side: float, out_size: int):
    arr = np.asarray(image)
    h, w = arr.shape[:2]

    left = cx - side / 2.0
    top = cy - side / 2.0
    right = left + side
    bottom = top + side

    pad_left = max(0, int(math.ceil(-left)))
    pad_top = max(0, int(math.ceil(-top)))
    pad_right = max(0, int(math.ceil(right - w)))
    pad_bottom = max(0, int(math.ceil(bottom - h)))

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.pad(arr, ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)), mode='edge')
        left += pad_left
        top += pad_top

    left_i = int(round(left))
    top_i = int(round(top))
    side_i = max(2, int(round(side)))
    crop = arr[top_i: top_i + side_i, left_i: left_i + side_i]
    crop_img = Image.fromarray(crop).resize((out_size, out_size), Image.Resampling.BILINEAR)
    crop_info = {'left': left_i - pad_left, 'top': top_i - pad_top, 'side': float(side_i), 'out': out_size}
    return crop_img, crop_info


def box_to_crop_normalized(box, crop_info):
    x, y, w, h = box
    scale = crop_info['out'] / crop_info['side']
    x = (x - crop_info['left']) * scale
    y = (y - crop_info['top']) * scale
    w = w * scale
    h = h * scale
    return [
        float((x + w / 2.0) / crop_info['out']),
        float((y + h / 2.0) / crop_info['out']),
        float(w / crop_info['out']),
        float(h / crop_info['out']),
    ]


def crop_normalized_to_box(box, crop_info, image_size):
    cx_n, cy_n, w_n, h_n = box
    out = float(crop_info['out'])
    cx = cx_n * out
    cy = cy_n * out
    w = max(2.0, w_n * out)
    h = max(2.0, h_n * out)
    scale = crop_info['side'] / crop_info['out']
    x = crop_info['left'] + (cx - w / 2.0) * scale
    y = crop_info['top'] + (cy - h / 2.0) * scale
    return clip_box_xywh([x, y, w * scale, h * scale], width=image_size[0], height=image_size[1])


def restore_frame(image: Image.Image, condition: str):
    bgr = cv2.cvtColor(np.asarray(image), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.2, tileGridSize=(8, 8)).apply(l)
    merged = cv2.merge([l, a, b])
    enhanced = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)
    if condition == 'rain':
        enhanced = cv2.medianBlur(enhanced, 3)
    else:
        enhanced = cv2.bilateralFilter(enhanced, 5, 35, 35)
    return Image.fromarray(cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB))


def bbox_iou(a, b):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    ax2, ay2 = ax + aw, ay + ah
    bx2, by2 = bx + bw, by + bh
    ix1, iy1 = max(ax, bx), max(ay, by)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    union = aw * ah + bw * bh - inter + 1e-6
    return float(inter / union)


def center_error(a, b):
    acx = a[0] + a[2] / 2.0
    acy = a[1] + a[3] / 2.0
    bcx = b[0] + b[2] / 2.0
    bcy = b[1] + b[3] / 2.0
    return float(((acx - bcx) ** 2 + (acy - bcy) ** 2) ** 0.5)

In [ ]:
class TrackingPairDataset(Dataset):
    def __init__(self, annotation_name='ExtremeTrack_train.json', condition_filter='all', samples_per_epoch=4000):
        self.records = load_sequences(annotation_name, condition_filter)
        self.samples_per_epoch = samples_per_epoch
        self.template_size = 128
        self.search_size = 256
        self.template_scale = 2.0
        self.search_scale = 4.5
        self.max_frame_gap = 12

    def __len__(self):
        return self.samples_per_epoch

    def _open(self, rel_path):
        return Image.open(DATASET_ROOT / rel_path).convert('RGB')

    def __getitem__(self, idx):
        rec = random.choice(self.records)
        end_idx = len(rec.image_paths) - 1
        s_idx = random.randint(1, end_idx)
        t_idx = random.randint(max(0, s_idx - self.max_frame_gap), s_idx - 1)

        t_img = self._open(rec.image_paths[t_idx])
        s_img = self._open(rec.image_paths[s_idx])
        t_box = [float(v) for v in rec.gt_rect[t_idx]]
        s_box = [float(v) for v in rec.gt_rect[s_idx]]

        t_cx, t_cy, t_side = compute_square_crop(t_box, self.template_scale)
        s_cx, s_cy, s_side = compute_square_crop(t_box, self.search_scale)

        t_crop, _ = crop_square(t_img, t_cx, t_cy, t_side, self.template_size)
        s_crop, s_info = crop_square(s_img, s_cx, s_cy, s_side, self.search_size)
        t_rest = restore_frame(t_crop, rec.condition)
        s_rest = restore_frame(s_crop, rec.condition)

        target = torch.tensor(box_to_crop_normalized(s_box, s_info), dtype=torch.float32)
        cid = 0 if rec.condition == 'haze' else 1
        return {
            'template': TF.to_tensor(t_crop),
            'template_restored': TF.to_tensor(t_rest),
            'search': TF.to_tensor(s_crop),
            'search_restored': TF.to_tensor(s_rest),
            'target_box': target,
            'condition': torch.tensor(cid, dtype=torch.long),
        }


class ConvStem(nn.Module):
    def __init__(self, dim=192):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(6, 64, 5, stride=2, padding=2, bias=False), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 96, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(96), nn.GELU(),
            nn.Conv2d(96, 128, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, dim, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(dim), nn.GELU(),
        )
    def forward(self, raw, restored):
        return self.net(torch.cat([raw, restored], dim=1))


class MixformerBlock(nn.Module):
    def __init__(self, dim=192, heads=6):
        super().__init__()
        self.n1 = nn.LayerNorm(dim)
        self.self_attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.n2 = nn.LayerNorm(dim)
        self.cross_attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.n3 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))
    def forward(self, t, s):
        s = s + self.self_attn(self.n1(s), self.n1(s), self.n1(s), need_weights=False)[0]
        s = s + self.cross_attn(self.n2(s), t, t, need_weights=False)[0]
        s = s + self.mlp(self.n3(s))
        return s


class MixformerTracker(nn.Module):
    def __init__(self, dim=192, depth=4, heads=6, cond_dim=16):
        super().__init__()
        self.ts = ConvStem(dim)
        self.ss = ConvStem(dim)
        self.blocks = nn.ModuleList([MixformerBlock(dim, heads) for _ in range(depth)])
        self.cond = nn.Embedding(2, cond_dim)
        self.head = nn.Sequential(nn.LayerNorm(dim + cond_dim), nn.Linear(dim + cond_dim, dim), nn.GELU(), nn.Linear(dim, 128), nn.GELU())
        self.box = nn.Linear(128, 4)
        self.score = nn.Linear(128, 1)
    def forward(self, t_raw, t_rest, s_raw, s_rest, c):
        tf = self.ts(t_raw, t_rest)
        sf = self.ss(s_raw, s_rest)
        t = tf.flatten(2).transpose(1, 2)
        s = sf.flatten(2).transpose(1, 2)
        for blk in self.blocks:
            s = blk(t, s)
        pooled = s.mean(dim=1)
        f = torch.cat([pooled, self.cond(c)], dim=1)
        h = self.head(f)
        return torch.sigmoid(self.box(h)), self.score(h).squeeze(-1)


def box_iou_tensor(pred, target):
    px1 = pred[:, 0] - pred[:, 2] / 2
    py1 = pred[:, 1] - pred[:, 3] / 2
    px2 = pred[:, 0] + pred[:, 2] / 2
    py2 = pred[:, 1] + pred[:, 3] / 2
    tx1 = target[:, 0] - target[:, 2] / 2
    ty1 = target[:, 1] - target[:, 3] / 2
    tx2 = target[:, 0] + target[:, 2] / 2
    ty2 = target[:, 1] + target[:, 3] / 2
    ix1 = torch.maximum(px1, tx1)
    iy1 = torch.maximum(py1, ty1)
    ix2 = torch.minimum(px2, tx2)
    iy2 = torch.minimum(py2, ty2)
    inter = (ix2 - ix1).clamp(min=0) * (iy2 - iy1).clamp(min=0)
    pa = (px2 - px1).clamp(min=1e-6) * (py2 - py1).clamp(min=1e-6)
    ta = (tx2 - tx1).clamp(min=1e-6) * (ty2 - ty1).clamp(min=1e-6)
    return inter / (pa + ta - inter + 1e-6)

In [ ]:
# ------------------------------
# Training config
# ------------------------------
CFG = {
    'condition': 'all',
    'epochs': 1,
    'batch_size': 24,
    'num_workers': 4,
    'samples_per_epoch': 3000,
    'lr': 2e-4,
    'weight_decay': 1e-4,
    'use_amp': True,
    'channels_last': True,
    'max_train_batches': 0,
    'max_val_sequences': 0,
}

train_ds = TrackingPairDataset('ExtremeTrack_train.json', condition_filter=CFG['condition'], samples_per_epoch=CFG['samples_per_epoch'])
val_seqs = load_sequences('ExtremeTrack_val.json', condition_filter=CFG['condition'])
if CFG['max_val_sequences'] > 0:
    val_seqs = val_seqs[:CFG['max_val_sequences']]

train_loader = DataLoader(
    train_ds,
    batch_size=CFG['batch_size'],
    shuffle=False,
    num_workers=CFG['num_workers'],
    pin_memory=True,
    persistent_workers=CFG['num_workers'] > 0,
)

model = MixformerTracker(dim=192, depth=4, heads=6, cond_dim=16).to(DEVICE)
if CFG['channels_last']:
    model = model.to(memory_format=torch.channels_last)

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scaler = GradScaler(device=DEVICE.type, enabled=CFG['use_amp'])
reg = nn.SmoothL1Loss()
bce = nn.BCEWithLogitsLoss()


def train_one_epoch(epoch):
    model.train()
    losses = []
    iterator = train_loader
    if CFG['max_train_batches'] > 0:
        iterator = []
        for i, b in enumerate(train_loader):
            iterator.append(b)
            if i + 1 >= CFG['max_train_batches']:
                break

    for batch in tqdm(iterator, desc=f'train epoch {epoch}'):
        t = batch['template'].to(DEVICE, non_blocking=True)
        tr = batch['template_restored'].to(DEVICE, non_blocking=True)
        s = batch['search'].to(DEVICE, non_blocking=True)
        sr = batch['search_restored'].to(DEVICE, non_blocking=True)
        y = batch['target_box'].to(DEVICE, non_blocking=True)
        c = batch['condition'].to(DEVICE, non_blocking=True)

        if CFG['channels_last']:
            t = t.contiguous(memory_format=torch.channels_last)
            tr = tr.contiguous(memory_format=torch.channels_last)
            s = s.contiguous(memory_format=torch.channels_last)
            sr = sr.contiguous(memory_format=torch.channels_last)

        with autocast(device_type=DEVICE.type, enabled=CFG['use_amp']):
            pb, ps_logit = model(t, tr, s, sr, c)
            iou = box_iou_tensor(pb, y).detach()
            st = (iou > 0.3).float()
            l_box = reg(pb, y) + (1.0 - box_iou_tensor(pb, y).mean())
            l_score = bce(ps_logit, st)
            loss = l_box + 0.25 * l_score

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        losses.append(float(loss.item()))

    return float(np.mean(losses)) if losses else 0.0


def validate_qp(iqa_cache_path=OUTPUT_ROOT / 'cache' / 'val_iqa_cache.json'):
    model.eval()
    br = BRISQUE(url=False)
    cache = {}
    if iqa_cache_path.exists():
        try:
            cache = load_json(iqa_cache_path)
        except Exception:
            cache = {}

    def get_iqa(path_str):
        if path_str in cache:
            return float(cache[path_str])
        arr = np.asarray(Image.open(path_str).convert('RGB'))
        b = float(br.score(arr))
        v = float(np.clip(1.0 - b / 100.0, 0.0, 1.0))
        cache[path_str] = v
        return v

    metrics = []
    preds_payload = {}

    with torch.no_grad():
        for seq in tqdm(val_seqs, desc='validate'):
            img0 = Image.open(DATASET_ROOT / seq.image_paths[0]).convert('RGB')
            cur = [float(v) for v in seq.gt_rect[0]]
            cond = torch.tensor([0 if seq.condition == 'haze' else 1], dtype=torch.long, device=DEVICE)

            tcx, tcy, ts = compute_square_crop(cur, 2.0)
            tc, _ = crop_square(img0, tcx, tcy, ts, 128)
            trc = restore_frame(tc, seq.condition)
            t = TF.to_tensor(tc).unsqueeze(0).to(DEVICE)
            tr = TF.to_tensor(trc).unsqueeze(0).to(DEVICE)

            pred_boxes = [cur.copy()]
            confs = [1.0]

            for fi in range(1, len(seq.image_paths)):
                img = Image.open(DATASET_ROOT / seq.image_paths[fi]).convert('RGB')
                scx, scy, ss = compute_square_crop(cur, 4.5)
                sc, sinfo = crop_square(img, scx, scy, ss, 256)
                src = restore_frame(sc, seq.condition)
                s = TF.to_tensor(sc).unsqueeze(0).to(DEVICE)
                sr = TF.to_tensor(src).unsqueeze(0).to(DEVICE)

                with autocast(device_type=DEVICE.type, enabled=CFG['use_amp']):
                    pbn, psl = model(t, tr, s, sr, cond)
                pb = crop_normalized_to_box(pbn[0].detach().cpu().tolist(), sinfo, img.size)
                ps = float(torch.sigmoid(psl)[0].item())

                if ps < 0.35:
                    blend = 0.65 * ps
                    pb = clip_box_xywh([
                        (1.0 - blend) * cur[0] + blend * pb[0],
                        (1.0 - blend) * cur[1] + blend * pb[1],
                        (1.0 - blend) * cur[2] + blend * pb[2],
                        (1.0 - blend) * cur[3] + blend * pb[3],
                    ], width=img.size[0], height=img.size[1])

                cur = pb
                pred_boxes.append(cur.copy())
                confs.append(ps)

                if ps > 0.6 and fi % 15 == 0:
                    tcx, tcy, ts = compute_square_crop(cur, 2.0)
                    tc, _ = crop_square(img, tcx, tcy, ts, 128)
                    trc = restore_frame(tc, seq.condition)
                    t = TF.to_tensor(tc).unsqueeze(0).to(DEVICE)
                    tr = TF.to_tensor(trc).unsqueeze(0).to(DEVICE)

            gt = [[float(v) for v in b] for b in seq.gt_rect]
            ious = np.array([bbox_iou(p, g) for p, g in zip(pred_boxes, gt)], dtype=np.float32)
            ces = np.array([center_error(p, g) for p, g in zip(pred_boxes, gt)], dtype=np.float32)
            iqas = np.array([get_iqa(str(DATASET_ROOT / p)) for p in seq.image_paths], dtype=np.float32)

            precision_20 = float((ces <= 20.0).mean())
            thresholds = np.linspace(0.0, 1.0, 21, dtype=np.float32)
            success_curve = np.array([(ious >= t).mean() for t in thresholds], dtype=np.float32)
            success_auc = float(success_curve.mean())
            qp = float(((iqas * ces) < 15.0).mean())

            metrics.append({
                'mean_iou': float(ious.mean()),
                'precision_20': precision_20,
                'success_auc': success_auc,
                'mean_iqa': float(iqas.mean()),
                'qp': qp,
            })

            preds_payload[seq.video_dir] = {
                'pred_rect': [[round(v, 3) for v in b] for b in pred_boxes],
                'confidence': [round(v, 5) for v in confs],
            }

    save_json(iqa_cache_path, cache)
    agg = {k: float(np.mean([m[k] for m in metrics])) for k in metrics[0].keys()}
    return agg, preds_payload


best_qp = -1.0
best_ckpt = None
history = []

for ep in range(1, CFG['epochs'] + 1):
    t0 = time.time()
    train_loss = train_one_epoch(ep)
    val_metrics, val_preds = validate_qp()
    elapsed = time.time() - t0

    ckpt_name = f"mixformer_kaggle_epoch{ep:02d}_iou{val_metrics['mean_iou']:.4f}_qp{val_metrics['qp']:.4f}.pt"
    ckpt_path = OUTPUT_ROOT / 'checkpoints' / ckpt_name
    torch.save({
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'epoch': ep,
        'metrics': val_metrics,
    }, ckpt_path)

    pred_path = OUTPUT_ROOT / 'predictions' / f'mixformer_kaggle_epoch{ep:02d}_val_predictions.json'
    save_json(pred_path, val_preds)

    row = {
        'epoch': ep,
        'train_loss': float(train_loss),
        **val_metrics,
        'checkpoint': str(ckpt_path),
        'prediction_file': str(pred_path),
        'epoch_seconds': round(elapsed, 2),
    }
    history.append(row)
    print(row)

    if val_metrics['qp'] > best_qp:
        best_qp = val_metrics['qp']
        best_ckpt = ckpt_path

    gc.collect()
    torch.cuda.empty_cache()

summary = {
    'best_qp': best_qp,
    'best_checkpoint': str(best_ckpt) if best_ckpt else '',
    'epochs': CFG['epochs'],
    'condition': CFG['condition'],
    'dataset_root': str(DATASET_ROOT),
}
save_json(OUTPUT_ROOT / 'experiments' / 'mixformer_kaggle_history.json', {'history': history, 'summary': summary})
save_json(OUTPUT_ROOT / 'leaderboard.json', {'mixformer_kaggle': summary})

print('\nDone')
print('Summary:', summary)
print('Output folder:', OUTPUT_ROOT)